The baseline LR ([`baseline_model.qmd`](baseline_model.qmd)) raised the
same diagnostic questions the EastAK baseline did: `*_count_5km` features
carrying top weights (exploration-density proxy), and whether SHAP tells
a geologically coherent story. This notebook runs the same RF + SHAP
analysis on BCGT that the EastAK [`random_forest_and_shap.qmd`](../eastak/random_forest_and_shap.qmd)
runs on its data.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from ai_minerals.data._common import DATA_DERIVED
from ai_minerals.grid import build_grid
from ai_minerals.model import (
    non_feature_columns, add_lithology_onehot, build_training_set,
    sample_pseudo_negatives, spatial_block_scores,
)
from ai_minerals.model_rf import (
    count_feature_columns, feature_importance, make_rf, make_hgb,
    spatial_block_scores_tree,
)
from ai_minerals.regions.bcgt import BCGT

df = pd.read_parquet(DATA_DERIVED / "features_bcgt_500m.parquet")
label_cols = tuple(f"is_{k}" for k in BCGT.deposit_classes)
df["is_any_deposit"] = (df[list(label_cols)].sum(axis=1) > 0).astype(np.uint8)
top_classes = df["lithology_class"].value_counts().head(10).index.tolist()
print(f"feature frame: {df.shape}  |  positives: {int(df['is_any_deposit'].sum())}")

## 1. The count-feature confound (BCGT edition)

Same trap as EastAK: the exploration-density proxies
(`*_count_5km`, `*_has_data_5km`) get top RF importance if they're in the
feature matrix. Fix: drop them.

In [ ]:
X, y = build_training_set(
    df, top_classes, n_per_positive=30, random_state=42,
    label_col="is_any_deposit",
    label_cols=label_cols + ("is_any_deposit",),
)
drop_cols = count_feature_columns(list(X.columns))
X_trim = X.drop(columns=drop_cols)
print(f"Full-feature training: {X.shape}")
print(f"Count-free training:   {X_trim.shape}  (dropped {len(drop_cols)} cols)")

rf_full = make_rf(); rf_full.fit(X.fillna(-9999), y)
rf_trim = make_rf(); rf_trim.fit(X_trim.fillna(-9999), y)

In [ ]:
imp_full = feature_importance(rf_full, list(X.columns)).head(10)
imp_trim = feature_importance(rf_trim, list(X_trim.columns)).head(10)
print("=== RF with count features — top 10 ===")
print(imp_full.to_string(index=False))
print("\n=== RF without count features — top 10 ===")
print(imp_trim.to_string(index=False))

## 2. Spatial-CV comparison

In [ ]:
negs = sample_pseudo_negatives(df, n_per_positive=30, random_state=42, label_col="is_any_deposit")
rows = pd.concat(
    [df[df["is_any_deposit"]==1][["row","col","x","y"]], negs[["row","col","x","y"]]],
    ignore_index=True,
)

lr_cv       = spatial_block_scores(X, y, rows, block_size_m=20_000.0)
rf_cv       = spatial_block_scores_tree(X, y, rows, model_factory=make_rf)
rf_trim_cv  = spatial_block_scores_tree(X_trim, y, rows, model_factory=make_rf)
hgb_trim_cv = spatial_block_scores_tree(X_trim, y, rows, model_factory=make_hgb)

print(f"{'model':<40} {'ROC-AUC ± sd':<18} {'PR-AUC ± sd':<18}")
for label, cv in [
    ("LR baseline (all features)          ", lr_cv),
    ("RF (all features)                    ", rf_cv),
    ("RF (no count / has_data features)    ", rf_trim_cv),
    ("HistGradientBoosting (no count)      ", hgb_trim_cv),
]:
    v = cv.dropna(subset=["roc_auc"])
    print(f"  {label}  {v['roc_auc'].mean():.3f} ± {v['roc_auc'].std():.3f}   {v['pr_auc'].mean():.3f} ± {v['pr_auc'].std():.3f}")

## 3. SHAP on the Random Forest (no count features)

In [ ]:
sv_path = DATA_DERIVED / "shap_rf_bcgt.npz"
if sv_path.exists():
    pack = np.load(sv_path, allow_pickle=True)
    sv = pack["sv"]
    feat_names = pack["feature_names"].tolist()
else:
    expl = shap.TreeExplainer(rf_trim)
    raw = expl.shap_values(X_trim.fillna(-9999))
    sv = raw[:, :, 1] if getattr(raw, "ndim", 0) == 3 else raw[1]
    feat_names = list(X_trim.columns)
    np.savez(sv_path, sv=sv, feature_names=np.array(feat_names), y=y)

mean_abs = np.abs(sv).mean(axis=0)
shap_df = pd.DataFrame({"feature": feat_names, "mean_abs_shap": mean_abs}).sort_values(
    "mean_abs_shap", ascending=False
).reset_index(drop=True)
print("Top-15 features by mean |SHAP|:")
print(shap_df.head(15).to_string(index=False))

In [ ]:
top = shap_df.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top["feature"], top["mean_abs_shap"])
ax.set_xlabel("mean |SHAP value|")
ax.set_title("BCGT top 15 features (SHAP, RF no-count, 4-class union)")
plt.tight_layout()

## 4. Directional SHAP — which direction does each feature push?

In [ ]:
pos_i = np.where(y == 1)[0]; neg_i = np.where(y == 0)[0]
cmp = pd.DataFrame({
    "feature": feat_names,
    "SHAP_at_pos": sv[pos_i].mean(axis=0),
    "SHAP_at_neg": sv[neg_i].mean(axis=0),
})
cmp["diff"] = cmp["SHAP_at_pos"] - cmp["SHAP_at_neg"]
print(cmp.reindex(cmp["diff"].abs().sort_values(ascending=False).index).head(12).to_string(index=False))

**Interpretation.** For a pooled-all-classes model, we expect a mix of:

- porphyry halo: Cu / Mo / Ag `*_max_5km` elevated
- epithermal pathfinders: Hg / Tl / As / Sb `*_max_5km` elevated
- VMS: Zn / Pb `*_max_5km` elevated
- magnetic contrast (NRCan 200 m)

If some of those deposit-type pathfinders land in the top-15 with
`SHAP_at_pos > 0` and `SHAP_at_neg < 0`, the model's learned a
multi-system halo signal. Features that flip sign at positives vs
negatives are the clearest directional signals.

## 5. AOI-wide prediction + capture rate

In [ ]:
all_rows = add_lithology_onehot(df, top_classes)
non_feat = non_feature_columns(label_cols + ("is_any_deposit",))
X_all = all_rows.drop(columns=[c for c in all_rows.columns if c in non_feat] + ["lithology_class"])
X_all_trim = X_all.drop(columns=drop_cols)

proba = rf_trim.predict_proba(X_all_trim.fillna(-9999))[:, 1]
is_pos = df["is_any_deposit"].to_numpy()
order = np.argsort(-proba)
total_pos = int(is_pos.sum())

print(f"=== Top-N% capture, BCGT (RF, no count/has_data; {total_pos} positives) ===")
for pct in (0.5, 1, 2, 5, 10, 20):
    k = int(pct / 100 * len(proba))
    cap = int(is_pos[order][:k].sum())
    print(f"  top {pct:>4.1f}% (k={k:>6,}) → {cap:3}/{total_pos}  ({100*cap/total_pos:4.0f}%)")

In [ ]:
grid = build_grid(BCGT.aoi, resolution_m=500, working_crs=BCGT.working_crs)
prob = np.full(grid.shape, np.nan, dtype=np.float32)
prob[df["row"].to_numpy(), df["col"].to_numpy()] = proba

fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(
    prob,
    extent=(grid.bounds[0], grid.bounds[2], grid.bounds[1], grid.bounds[3]),
    origin="lower", cmap="viridis", vmin=0, vmax=1,
)
plt.colorbar(im, ax=ax, label="P(any deposit class)")
colors = {"is_porphyry":"red","is_epithermal":"yellow","is_skarn":"cyan","is_vms":"magenta"}
for c, col in colors.items():
    p = df[df[c]==1]
    ax.scatter(p["x"], p["y"], s=20, marker="*", c=col, edgecolor="black", linewidth=0.3,
               label=f"{c} ({int(df[c].sum())})")
ax.set_title("Random Forest prospectivity — BCGT (no count/has_data features)")
ax.set_xlabel("Easting (m, EPSG:3005)"); ax.set_ylabel("Northing (m, EPSG:3005)")
ax.legend(loc="lower left", fontsize=8); ax.set_aspect("equal")
plt.tight_layout()

## Summary

- The count-feature confound is present in BCGT the same way it was in
  EastAK: `*_count_5km` and `*_has_data_5km` dominate RF importance if
  kept in the matrix. Dropping them shifts top features to `*_max_5km`
  of pathfinder elements — the honest signal.
- Pooled 4-class SHAP surfaces features from multiple deposit systems
  simultaneously; a per-class one-vs-rest variant would make clearer
  attributions per deposit type.
- Next: the external blind test against 366 post-2015 BC drill holes
  in [`validation.qmd`](validation.qmd).